# M7 · Ranking & CTR-family

_Curriculum · Domain 1 · Ranking & Recommenders_

**Turn response probabilities into an ordered list.**

We compare pointwise pCTR scoring, pairwise loss, and a simple multi-objective rank score. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - CPU-only and deterministic.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## First, look at candidates

Each row is an ad candidate with predicted probabilities. A ranker may combine $pCTR$, $pVTR$, and value weights rather than sorting by clicks alone.

In [ ]:
df = pd.DataFrame({"ad": ["A", "B", "C", "D"], "bid": [5.0, 8.0, 4.0, 6.0], "pctr": [0.026, 0.016, 0.030, 0.020], "pvtr": [0.18, 0.25, 0.10, 0.22], "clicked": [1, 0, 1, 0]})

print(df)

## The pairwise objective

For a clicked item $i^+$ and skipped item $i^-$, RankNet-style loss is

$$\ell=-\log\sigma(s_{i^+}-s_{i^-})$$

A larger positive margin means smaller loss.

### Step 1 - Build a pointwise score

Expected click value is a simple pointwise ranking score: bid times calibrated pCTR.

In [ ]:
df["click_value"] = df["bid"] * df["pctr"]
pointwise = df.sort_values("click_value", ascending=False)

print(pointwise[["ad", "click_value"]])

assert pointwise.iloc[0]["ad"] == "A"

### Step 2 - Compute one pairwise loss

Compare clicked ad A with skipped ad B using the pointwise score as $s$.

In [ ]:
s_pos = float(df.loc[df["ad"] == "A", "click_value"].iloc[0])
s_neg = float(df.loc[df["ad"] == "B", "click_value"].iloc[0])
margin = s_pos - s_neg
prob_order = 1.0 / (1.0 + np.exp(-margin))
pair_loss = -np.log(prob_order)

print("margin:", round(margin, 4))
print("pairwise loss:", round(pair_loss, 4))

assert pair_loss < 0.75

### Step 3 - Add a video-view head

A multi-objective score can include both click value and video value, as long as the heads are meaningful.

In [ ]:
df["multi_score"] = df["click_value"] + 0.25 * df["pvtr"]
multi = df.sort_values("multi_score", ascending=False)

print(multi[["ad", "click_value", "pvtr", "multi_score"]].round(4))

assert set(multi["ad"]) == set(df["ad"])

## Visualize score components

The plot makes the trade-off visible: click value and video-view value can disagree.

In [ ]:
x = np.arange(len(df))
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(x - 0.18, df["click_value"], width=0.36, label="click value")
ax.bar(x + 0.18, 0.25 * df["pvtr"], width=0.36, label="video term")
ax.set_xticks(x)
ax.set_xticklabels(df["ad"])
ax.set_title("multi-objective rank terms")
ax.legend()
plt.show()

## Practice

Try each in the empty cell below it.

1. Change the video weight from 0.25 to 0.10 and inspect the top ad.
2. Compute pairwise loss for clicked C versus skipped D.
3. Replace `bid * pctr` with pCTR-only ranking and compare the order.

In [ ]:
# Your turn:
